手書き文字データは一つの文字が
$(x_1,x_2, x_{64})$
で表される64次元ベクトルである。

多数の画像を集めると、
それらは64次元空間上の「分布」を形成する。

例えば数字「0」の画像群、
数字「1」の画像群は
それぞれ異なる「分布」を持つ。





分布を実感してもらうために、
$p(z_1,z_2∣digit)$をPCA二次元で示す。

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# --
# data
# --
digits = load_digits()

x = digits.data / 16.0
y = digits.target

# --
# PCA
# --
pca = PCA(n_components=2)
z = pca.fit_transform(x)

# 共通軸範囲
xmin = z[:,0].min()
xmax = z[:,0].max()

ymin = z[:,1].min()
ymax = z[:,1].max()

# --
# plot
# --
fig, axes = plt.subplots(
    2,
    5,
    figsize=(15,6),
    sharex=True,
    sharey=True
)

axes = axes.ravel()

for digit in range(10):

    ax = axes[digit]

    idx = (y == digit)

    ax.scatter(
        z[idx,0],
        z[idx,1],
        s=10,
        alpha=0.5
    )

    center = z[idx].mean(axis=0)

    ax.scatter(
        center[0],
        center[1],
        marker="x",
        s=200
    )

    ax.set_title(f"Digit {digit}")

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    ax.grid(True)

fig.supxlabel("PCA1")
fig.supylabel("PCA2")
fig.suptitle("PCA 2D Distribution for Each Digit")

plt.tight_layout()
plt.show()



手書き文字 Autoencoder を用いると、64次元の画像データを2次元の latent 空間へ圧縮できる。

このとき、各手書き文字は latent 空間上で特徴的な分布を形成する。


一方、2次元 Gaussian 分布

$$
z \sim \mathcal N(0,I)
$$

は単純な円形の分布である。

生成モデルの役割は、この単純な Gaussian 分布を、手書き文字データが持つ複雑な分布へ変換することである。


Diffusion Model や Flow Matching は、いずれもこの分布変換を実現するための手法である。




In [ ]:
# 初期値である
# ２D Gaussian分布を次に示す。

import numpy as np
import matplotlib.pyplot as plt

# --
# sample from N(0,I)
# --

n = 1000

x = np.random.randn(n)
y = np.random.randn(n)

# --
# plot
# --

plt.figure(figsize=(3,3))

plt.scatter(
    x,
    y,
    s=3,
    alpha=0.2
)

plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")
plt.title("2D Gaussian Samples")

plt.show()

# 0. 表記定義

機械学習、統計では

$$ x\sim q(x)$$

という表記を用いる。これは確率変数xが分布qに従う、という意味。

厳密には、
$$ x\sim q$$
と書く。


# 1. 生成モデルとは何か

例えば結晶構造、分子構造、画像などの実データは、

$$
x \sim q(x)
$$

という複雑な分布に従っています。

生成モデルの目的は

$$
q(x)
$$

を再現することです。



例えば

* 結晶
* 電子密度
* 手書き数字画像

などを大量に集めると、高次元空間のごく一部、ラベル毎にごく一部に分布しています。

生成モデルは

**「その分布そのものを学習する」**

ことが目的です。


以下では、
diffusion modelもflow matchingも
$$
\mathbb{R}^d \rightarrow \mathbb{R}^d
$$
の変換を行ないます。


---

# 2. Diffusion Model の考え方



## 前向き過程



まずデータに少しずつノイズを加えます。

$$
x_0^{data}
\rightarrow
x_1
\rightarrow
x_2
\rightarrow
\cdots
\rightarrow
x_T
$$

最終的には

$$
x_T \sim N(0,I)
$$

になります。 


式では

$$
x_t
=
\sqrt{\bar{\alpha}_t}x_0
+
\sqrt{1-\bar{\alpha}_t}\epsilon
$$

$$
\epsilon\sim N(0,I)
$$

です。 



## 逆過程

今度は

$$
x_T
\rightarrow
x_{T-1}
\rightarrow
\cdots
\rightarrow
x_0^{data, prediction}
$$

を行います。

つまり

ノイズからデータを復元します。



しかし本当の逆過程は分からないので、

ニューラルネットに

$$
\epsilon_\theta(x_t,t)
$$

を学習させます。





# 4. Flow Matching の考え方

かなり簡単には：
Diffusion modelは「差分」だった。
→💡ならば微分にできるのでは、という考えか方。




ガウシアンノイズ分布
から
データ分布
への
**流れ（Flow）**
を直接学習します。 



# 5. Flow Matching の学習

最も単純な場合、

ノイズ

$$
x_0
$$

と

データ

$$
x_1
$$

を結ぶ直線を考えます。(diffusion modelとは$t=0$, $t=1$の方向が反対なことに注意）

$$
x_t
=
(1-t)x_0+tx_1
$$

です。 



このとき速度は

$$
u
=
x_1-x_0
$$

です。



ニューラルネットに

$$
u_\theta(x,t)
$$

を学習させます。


---

# 6. 両者の違い


## Diffusion 

forward:
```
Data
 ↓
少しずつノイズ追加
 ↓
Gaussian Noise
```

backward:
```
Gaussian Noise
 ↓
少しずつ除去
 ↓
Data
```



## Flow Matching

```
Gaussian Noise
    ↓
 速度場
    ↓
 Data
```


# 7. 理論の関係

最近の理解では


```
確率的Diffusion (DDPM）
        ↓
決定論的 Diffusion (Probability Flow ODE , DDIM)
        ↓
Flow Matching（速度場学習）
```

という連続的な関係があります。


## 用途

近年は画像生成AIでも

- Flow matching: Stable Diffusion 3
- Diffusion: DALLE 3
  
となっており、Flow matchingでも高品質な画像生成が可能になっている。

→物理・科学系では、解釈や速度で決める。

protein構造生成例：
- SE(3) Diffusion: arXiv:2203.02923, GeoDiff
- SE(3) Flow matching: arXiv:2310.02391, SE(3)-Stochastic Flow Matching

最近はprotein構造生成はFlow matchingの方が流行っているらしい。

### E(3)とSE(3)
- E(3):translation, rotation, inversion
- SE(3): translation, rotation ,inversion, refletion


# 8. まとめ

- Diffusion：forward diffusion を明示的に定義する
- Flow Matching：forward diffusion の替わりに、補間経路（interpolation path, probability path）を定義し、その経路に沿う速度場を学習する。生成時には速度場を「時間」積分して Gaussian Noise を Data へ変換する。。

（線形）補間経路：
$$ x_t = (1-t)x_0 + t x_1 $$

# References

## diffusion model

arXiv:2312.10393v1,
Lecture Notes in Probabilistic Diffusion Models, Inga Strümke and Helge Langseth, Norwegian University of Science and Technology"

## flow matching



arXiv:2412.06264v1, Flow Matching Guide and Code,
Yaron Lipman, Marton Havasi, Peter Holderrieth, Neta Shaul, Matt Le, Brian Karrer, Ricky T. Q. Chen, David Lopez-Paz, Heli Ben-Hamu, Itai Gat.
Also, https://peterroelants.github.io/posts/flow_matching_intro/

arXiv:2506.02070v3, An Introduction to Flow Matching and Diffusion Models,
Peter Holderrieth, Ezra Erives.

